## **Lab de Evals: ¿cómo sé si mi sistema funciona?**

Sin evaluación automática, cada cambio de prompt o de modelo es una apuesta. En este lab construimos un **eval** completo
para el asistente RAG de **Bicis Nube** (el de la demo de la clase 1) y lo usamos para tomar decisiones con datos:

1. **Dataset**: 16 preguntas con la respuesta esperada (incluidas trampas que no están en los documentos).
2. **Ejecutar** el sistema sobre todas.
3. **Puntuar**: reglas en código + un **LLM-as-judge** con rúbrica.
4. **Comparar** variantes (prompt, nivel de razonamiento, modelo) en acierto, coste y latencia.

Y de paso comprobamos que **el juez también se equivoca**.

In [ ]:
!pip install -q --upgrade openai pandas numpy pydantic

In [ ]:
import os
from getpass import getpass

# La clave NUNCA se escribe en el notebook. Orden: secretos de Colab -> variable de entorno -> teclado
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")
print("Clave cargada:", os.environ["OPENAI_API_KEY"][:7] + "…")

La clave identifica tu cuenta y se factura contra ella. **Nunca** la escribas en el notebook: en Colab, guárdala en *Secretos* (icono de la llave) con el nombre `OPENAI_API_KEY`; en tu ordenador, como variable de entorno. Si no la encuentra, te la pide sin mostrarla.

In [ ]:
import time
from typing import Literal

import numpy as np
import pandas as pd
from openai import OpenAI
from pydantic import BaseModel, Field

client = OpenAI()
MODELO = "gpt-6-luna"
PRECIOS = {"gpt-6-luna": (0.10, 0.50), "gpt-6.1-terra": (1.00, 5.00)}   # $ por millón (entrada, salida), nov 2026
pd.set_option("display.max_colwidth", 70)

## 1. El sistema que vamos a evaluar

In [ ]:
documentos = [
    {"id": "tarifas-1", "texto": "Bicis Nube tiene tres planes: Básico (9 €/mes, 60 minutos al día), Plus (19 €/mes, minutos ilimitados) y Empresa (precio a medida, desde 10 bicis)."},
    {"id": "tarifas-2", "texto": "Sin plan, el alquiler cuesta 0,25 € por minuto más 1 € de desbloqueo. Los estudiantes con carné tienen un 30 % de descuento en el plan Plus."},
    {"id": "bateria-1", "texto": "Si una bici se devuelve con la batería por debajo del 10 %, se cobra una penalización de 3 €. Por encima del 50 % se regalan 10 minutos para el siguiente viaje."},
    {"id": "bateria-2", "texto": "Las bicis tienen 70 km de autonomía. La app avisa cuando quedan menos de 8 km y muestra la estación de carga más cercana."},
    {"id": "averias-1", "texto": "Para reportar una avería: botón 'Problema con la bici' en la app, foto obligatoria. El viaje se detiene y no se cobra desde ese momento."},
    {"id": "seguro-1", "texto": "Todos los viajes incluyen seguro de responsabilidad civil. El robo de la bici durante un viaje sin candado puesto tiene una franquicia de 150 €."},
    {"id": "empresa-1", "texto": "El plan Empresa incluye panel de control, facturación mensual única y mantenimiento en las oficinas del cliente cada 15 días."},
    {"id": "zonas-1", "texto": "El servicio funciona en Madrid, Valencia y Bilbao. Aparcar fuera de zona cuesta 5 € y la app bloquea el fin del viaje."},
]
emb = client.embeddings.create(model="text-embedding-3-small", input=[d["texto"] for d in documentos])
indice = np.array([e.embedding for e in emb.data])

def recuperar(pregunta, top_k=2):
    q = np.array(client.embeddings.create(model="text-embedding-3-small", input=[pregunta]).data[0].embedding)
    return [documentos[i] for i in np.argsort(-(indice @ q))[:top_k]]

PROMPT_BUENO = ("Eres el asistente de Bicis Nube. Responde SOLO con el CONTEXTO y cita el id entre corchetes, p. ej. [tarifas-1]. "
                "Si la respuesta no está en el contexto, responde exactamente: 'No tengo esa información'.")
PROMPT_FLOJO = "Eres el asistente de Bicis Nube. Usa el contexto para ayudar al cliente."

def asistente(pregunta, prompt=PROMPT_BUENO, modelo=MODELO, esfuerzo="none", top_k=2):
    """Devuelve la respuesta y las métricas de la llamada (coste y latencia)."""
    contexto = "\n".join(f"[{d['id']}] {d['texto']}" for d in recuperar(pregunta, top_k))
    t0 = time.time()
    r = client.responses.create(model=modelo, reasoning={"effort": esfuerzo},
                                instructions=f"{prompt}\n\nCONTEXTO:\n{contexto}", input=pregunta)
    p_in, p_out = PRECIOS[modelo]
    coste = (r.usage.input_tokens * p_in + r.usage.output_tokens * p_out) / 1e6
    return {"respuesta": r.output_text.strip(), "latencia_s": time.time() - t0, "coste_usd": coste}

print(asistente("¿Cuánto cuesta el plan Plus?")["respuesta"])

## 2. El dataset de evaluación
Cada caso tiene la pregunta, un **dato que debe aparecer** en la respuesta, la **fuente** que debería citar y si es una
**trampa** (la respuesta no está en los documentos y lo correcto es decir «No tengo esa información»).
En un proyecto real: 30-100 casos sacados de preguntas reales de usuarios y de fallos pasados.

In [ ]:
dataset = pd.DataFrame([
    # pregunta,                                                     dato esperado, fuente,      trampa
    ("¿Cuánto cuesta el plan Plus al mes?",                           "19",        "tarifas-1", False),
    ("¿Cuántos minutos al día incluye el plan Básico?",               "60",        "tarifas-1", False),
    ("¿Cuánto cuesta desbloquear una bici si no tengo plan?",         "1",         "tarifas-2", False),
    ("Soy estudiante, ¿qué descuento tengo en el Plus?",              "30",        "tarifas-2", False),
    ("¿Qué me cobran si devuelvo la bici casi sin batería?",          "3",         "bateria-1", False),
    ("¿Qué gano si la devuelvo con más de la mitad de carga?",        "10",        "bateria-1", False),
    ("¿Cuántos kilómetros de autonomía tiene la bici?",               "70",        "bateria-2", False),
    ("Se me ha roto la bici en mitad del viaje, ¿qué hago?",          "foto",      "averias-1", False),
    ("Me la han robado sin candado, ¿cuánto pago?",                   "150",       "seguro-1",  False),
    ("¿Cada cuánto viene el mantenimiento en el plan Empresa?",       "15",        "empresa-1", False),
    ("¿Cuánto me cuesta dejarla fuera de la zona de servicio?",       "5",         "zonas-1",   False),
    ("¿En qué ciudades está Bicis Nube?",                             "Bilbao",    "zonas-1",   False),
    ("¿Cuál es el teléfono de atención al cliente?",                  None,        None,        True),
    ("¿Tenéis bicis para niños?",                                     None,        None,        True),
    ("¿Puedo pagar con bitcoin?",                                     None,        None,        True),
    ("¿Cuál es el horario de la oficina central?",                    None,        None,        True),
], columns=["pregunta", "esperado", "fuente", "trampa"])
print(len(dataset), "casos ·", dataset.trampa.sum(), "trampas")
dataset.head()

## 3. Ejecutar el sistema sobre todo el dataset

In [ ]:
def ejecutar(**config):
    filas = [asistente(p, **config) for p in dataset.pregunta]
    return pd.concat([dataset, pd.DataFrame(filas)], axis=1)

res = ejecutar()
res[["pregunta", "respuesta"]].head(6)

## 4. Puntuar con reglas en código
Lo que se puede comprobar con código, se comprueba con código: es **gratis, rápido y determinista**.

In [ ]:
NO_SE = "no tengo esa información"

def puntuar_reglas(fila):
    r = fila.respuesta.lower()
    if fila.trampa:
        return pd.Series({"ok_dato": NO_SE in r, "ok_cita": NO_SE in r})
    return pd.Series({"ok_dato": str(fila.esperado).lower() in r,
                      "ok_cita": f"[{fila.fuente}]" in r})

res = pd.concat([res, res.apply(puntuar_reglas, axis=1)], axis=1)
print(f"Dato correcto: {res.ok_dato.mean():.0%} · Cita correcta: {res.ok_cita.mean():.0%}")
res[~(res.ok_dato & res.ok_cita)][["pregunta", "respuesta", "ok_dato", "ok_cita"]]

**Revisa los fallos a mano.** ¿Son errores del modelo, de la **regla** (p. ej. «1» también está dentro de «19») o de la **recuperación**?
Mira las respuestas: si dice «No tengo esa información» a una pregunta que sí está en los documentos, el problema suele ser que
`recuperar()` no trajo el fragmento correcto. El modelo hizo lo que le pedimos: no inventar.

## 5. LLM-as-judge con rúbrica
El juez recibe la pregunta, el contexto correcto y la respuesta, y puntúa la **fidelidad**: ¿todo lo que dice está respaldado por los documentos?

In [ ]:
class Juicio(BaseModel):
    fiel: bool = Field(description="True si TODO lo que afirma está en el contexto (o si dice que no tiene la información)")
    nota: int = Field(ge=1, le=5, description="5 = correcta, completa y citada; 1 = inventada o inútil")
    motivo: str

RUBRICA = ("Eres un evaluador estricto de un asistente de atención al cliente. Recibes los DOCUMENTOS de la empresa, la PREGUNTA "
           "y la RESPUESTA. Marca fiel=False si la respuesta afirma algo que no está en los documentos. "
           "Si la pregunta no se puede responder con los documentos, la respuesta correcta es decir que no se tiene la información (nota 5). "
           "Nota: 5 correcta y citada, 4 correcta sin cita, 3 incompleta, 2 con algún dato inventado, 1 inventada o inútil.")
TODOS = "\n".join(f"[{d['id']}] {d['texto']}" for d in documentos)

def juez(pregunta, respuesta, modelo=MODELO):
    return client.responses.parse(model=modelo, reasoning={"effort": "low"}, instructions=RUBRICA, text_format=Juicio,
                                  input=f"DOCUMENTOS:\n{TODOS}\n\nPREGUNTA: {pregunta}\nRESPUESTA: {respuesta}").output_parsed

juicios = [juez(p, r) for p, r in zip(res.pregunta, res.respuesta)]
res["fiel"] = [j.fiel for j in juicios]
res["nota"] = [j.nota for j in juicios]
res["motivo"] = [j.motivo for j in juicios]
print(f"Fidelidad: {res.fiel.mean():.0%} · Nota media: {res.nota.mean():.2f}/5")
res[["pregunta", "nota", "motivo"]].sort_values("nota").head(5)

## 6. Comparar variantes: aquí está el valor del eval
Ahora que medir es automático, probar cambios cuesta segundos. ¿Qué pasa con un prompt más flojo, con más razonamiento o con un modelo mayor?

In [ ]:
def evaluar(nombre, **config):
    r = ejecutar(**config)
    r = pd.concat([r, r.apply(puntuar_reglas, axis=1)], axis=1)
    j = [juez(p, x) for p, x in zip(r.pregunta, r.respuesta)]
    return {"variante": nombre,
            "dato_ok": r.ok_dato.mean(), "cita_ok": r.ok_cita.mean(),
            "trampas_ok": r[r.trampa].ok_dato.mean(),
            "fidelidad": np.mean([x.fiel for x in j]), "nota": np.mean([x.nota for x in j]),
            "latencia_s": r.latencia_s.mean(), "coste_1000_preg_usd": r.coste_usd.mean() * 1000}

variantes = [
    evaluar("base (luna, none)"),
    evaluar("prompt flojo", prompt=PROMPT_FLOJO),
    evaluar("luna + effort low", esfuerzo="low"),
    evaluar("terra (modelo mayor)", modelo="gpt-6.1-terra"),
    evaluar("pasar los 8 documentos", top_k=8),
]
tabla = pd.DataFrame(variantes).set_index("variante")
for c in ["dato_ok", "cita_ok", "trampas_ok", "fidelidad"]:
    tabla[c] = (tabla[c] * 100).round().astype(int).astype(str) + " %"
tabla.round({"nota": 2, "latencia_s": 1, "coste_1000_preg_usd": 3})

**Cómo leer la tabla** (en nuestra ejecución, tus números pueden variar un poco):
- El **prompt flojo** se hunde en citas y en **trampas**: sin «si no está, dilo», el modelo rellena huecos. ¡Y el juez le pone buena nota media! Por eso se miden **varias métricas**.
- **Más razonamiento** o un **modelo 10 veces más caro** no mejoran nada: el fallo no estaba en el modelo, sino en la **recuperación**.
- **Pasar los 8 documentos** sí lo arregla: las dos preguntas que fallaban no recuperaban su fragmento (la búsqueda semántica
  falla con preguntas cortas como «¿en qué ciudades está?»). Con 8 documentos cabe todo en el prompt; con 10.000, no:
  ahí toca mejorar la recuperación (búsqueda híbrida y re-ranking, en el **Lab de RAG avanzado**).
- Sin eval habríamos pagado un modelo 10 veces más caro para nada.
- Guarda este eval: cada vez que cambies algo, lo vuelves a pasar (**test de regresión**).

## 7. El juez también se equivoca
### 7.1 Sesgo de posición
Le enseñamos dos respuestas y le pedimos que elija la mejor. Luego **cambiamos el orden**. Un juez sin sesgo elegiría la misma.

In [ ]:
class Eleccion(BaseModel):
    mejor: Literal["A", "B"]

pregunta = "¿Qué hago si se me rompe la bici?"
r1 = "Pulsa 'Problema con la bici' en la app y sube una foto; el viaje se detiene y no se te cobra desde ese momento. [averias-1]"
r2 = "Usa el botón 'Problema con la bici' de la app, adjunta una foto obligatoria y el cobro se para en ese instante. [averias-1]"

def elegir(a, b):
    return client.responses.parse(model=MODELO, reasoning={"effort": "none"}, text_format=Eleccion,
                                  input=f"Pregunta: {pregunta}\nRespuesta A: {a}\nRespuesta B: {b}\n¿Cuál es mejor?").output_parsed.mejor

resultados = []
for _ in range(5):
    resultados.append(("r1" if elegir(r1, r2) == "A" else "r2", "r1" if elegir(r2, r1) == "B" else "r2"))
consistentes = sum(a == b for a, b in resultados)
print("Elegida (orden normal, orden invertido):", resultados)
print(f"Consistente en {consistentes}/5 · Primera posición elegida: "
      f"{sum(x[0]=='r1' for x in resultados) + sum(x[1]=='r2' for x in resultados)}/10 veces")

Las dos respuestas son prácticamente iguales. Lo que comprobamos es si el juez **mantiene su elección al darles la vuelta**.
Si cambia de opinión al cambiar el orden, tiene sesgo de posición. Los modelos actuales son bastante más consistentes que los de 2024,
pero el sesgo reaparece con respuestas de calidad parecida o de longitud distinta. **Remedio:** evaluar en los dos órdenes y quedarse solo con los acuerdos.

### 7.2 Calibrar con etiquetas humanas
Antes de fiarte de un juez, compáralo con **tus** notas en unos cuantos casos. Si no coincide, arregla la rúbrica.

In [ ]:
casos = [  # respuesta, ¿es fiel? (tu etiqueta)
    ("¿Cuánto cuesta el Plus?", "19 € al mes, minutos ilimitados. [tarifas-1]", True),
    ("¿Cuánto cuesta el Plus?", "19 € al mes y el primer mes es gratis. [tarifas-1]", False),          # inventa la promoción
    ("¿Tenéis bicis para niños?", "No tengo esa información.", True),
    ("¿Tenéis bicis para niños?", "Sí, tenemos modelos infantiles en Madrid.", False),
    ("¿Qué pasa si aparco fuera de zona?", "Te cobran 5 € y la app no te deja terminar el viaje. [zonas-1]", True),
    ("¿Qué pasa si aparco fuera de zona?", "Te cobran 5 €, salvo los fines de semana. [zonas-1]", False),  # excepción inventada
]
acuerdo = [juez(p, r).fiel == humano for p, r, humano in casos]
print(f"El juez coincide contigo en {sum(acuerdo)}/{len(casos)} casos")

Si coincide en (casi) todos, puedes usarlo para evaluar miles de respuestas. Si no, mira en qué falla y **ajusta la rúbrica**,
no tu criterio. Herramientas como DeepEval, Ragas, Langfuse o LangSmith automatizan todo esto, pero la idea es la misma
que acabas de programar en pandas.

### Reto
1. Añade 4 casos nuevos al dataset (2 normales y 2 trampas) y vuelve a pasar la tabla de variantes.
2. Arregla la recuperación sin pasar todos los documentos: añade a cada texto un título descriptivo
   (p. ej. «Ciudades y zonas de servicio: …»), vuelve a crear el índice y mide con `top_k=2`. Es la idea de la *contextual retrieval*.